# ClassTran: separate temporal and geographical flexibility indices

This notebook uses the same Ecolane reservation/trip workbook and the agreed demand segment:

$$
	ext{demand segment} =
	ext{Purpose} + 	ext{1.5-mile origin zone} + 	ext{weekday}.
$$

No ride ID or customer ID is part of the segment.

Workshop, Employment, Education, and Medical are policy-rigid. Their temporal and geographical flexibility indices are reported as **NA**, and they do not enter either actionable candidate pool.


## Index definitions

### Temporal flexibility

Promised Pick-up Time is divided into 30-minute, half-open bins. For example, 08:00 belongs to 08:00-08:30, 08:29 belongs to 08:00-08:30, and 08:30 belongs to 08:30-09:00.

For segment $s$ and time bin $t$:

$$p_{s,t}=rac{n_{s,t}}{N_s}, \qquad
t_s^*=rg\max_t p_{s,t}.$$

The alternative-time share is:

$$A_s=1-p_{s,t_s^*}.$$

The average alternative shift, conditional on observing a non-peak time, is:

$$D_s=
rac{\sum_{t
e t_s^*}p_{s,t}|t-t_s^*|}{A_s}.$$

With shift horizon $H=120$ minutes:

$$T_s=
\sum_{t
e t_s^*}
p_{s,t}\min\left(rac{|t-t_s^*|}{H},1ight).$$

$T_s$ increases when more demand is observed outside the peak and those alternatives are farther away. It is an **observed shift-potential index**, not proof that a particular rider will accept a shift.

### Geographical flexibility

For eligible segments, candidate destinations are observed 1.5-mile destination grid zones. If $p_{s,d}$ is destination $d$'s visit share:

$$G_s=1-\sum_d p_{s,d}^2.$$

The effective destination count is:

$$N_{\mathrm{effective}}=rac{1}{\sum_d p_{s,d}^2}.$$

The Gini-Simpson score summarizes diversity. A separate ranked candidate table identifies the destination with the highest visits and the share of every alternative.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

PROJECT_DIR = Path("/scratch/umni5/a/li5125/DOE_analysis/RFI-Rider_flexibility_index-")
DATA_PATH = PROJECT_DIR / "Ecolane Reservation and Trip Data July 2022 - June 2023.xlsx"
SHEET_NAME = "SMART Trip Data"

GRID_MILES = 1.5
TIME_BIN_MINUTES = 30
SHIFT_HORIZON_MINUTES = 120
MIN_SEGMENT_TRIPS = 10
MIN_SEGMENT_SERVICE_DAYS = 3


In [2]:
required_source_columns = [
    "Trip ID", "Trip Date", "Purpose", "Promised Pick-up Time",
    "Pick-up Latitude", "Pick-up Longitude",
    "Drop-off Latitude", "Drop-off Longitude",
]

df_raw = pd.read_excel(
    DATA_PATH,
    sheet_name=SHEET_NAME,
    usecols=required_source_columns,
)

print(f"Source: {DATA_PATH.name}")
print(f"Rows loaded: {len(df_raw):,}")
print(f"Date range: {pd.to_datetime(df_raw['Trip Date']).min().date()} to "
      f"{pd.to_datetime(df_raw['Trip Date']).max().date()}")
print("Purpose counts:")
display(df_raw["Purpose"].value_counts(dropna=False).rename("trips").to_frame())


Source: Ecolane Reservation and Trip Data July 2022 - June 2023.xlsx
Rows loaded: 121,281
Date range: 2022-07-01 to 2023-06-30
Purpose counts:


,trips
Purpose,
Nutrition,64285
Medical,17015
Employment,13619
Dialysis,13186
Workshop,5511
Personal,2689
Shopping,2192
Education,1526
Recreation,1001


## Calculation functions

The following cell contains the complete implementation so that the notebook is self-contained.


In [3]:
"""Separate temporal and geographical flexibility indices for ClassTran.

Demand segment: Purpose + 1.5-mile origin grid zone + weekday.
"""

from __future__ import annotations

import numpy as np
import pandas as pd


RIGID_PURPOSES = {"Workshop", "Employment", "Education", "Medical"}


def _clean_purpose(series):
    return (
        series.astype("string").str.strip().replace("", pd.NA)
        .fillna("Missing / Unknown")
    )


def _time_minutes(series):
    """Convert Excel/Python clock-time values to minutes after midnight."""
    text = series.astype("string").str.strip()
    parsed = pd.to_datetime(text, format="%H:%M:%S", errors="coerce")
    missing = parsed.isna()
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(
            text.loc[missing], format="%H:%M", errors="coerce"
        )
    return parsed.dt.hour * 60 + parsed.dt.minute + parsed.dt.second / 60


def _clock_label(minutes):
    minutes = int(minutes) % 1440
    return f"{minutes // 60:02d}:{minutes % 60:02d}"


def _interval_label(start_minutes, width):
    return f"{_clock_label(start_minutes)}-{_clock_label(start_minutes + width)}"


def _make_grid_zone(lat, lon, lat0, lon0, lat_step, lon_step):
    lat_index = np.floor((lat.to_numpy(dtype=float) - lat0) / lat_step).astype(int)
    lon_index = np.floor((lon.to_numpy(dtype=float) - lon0) / lon_step).astype(int)
    return pd.Series(
        "r" + pd.Series(lat_index, index=lat.index).astype(str)
        + "_c" + pd.Series(lon_index, index=lon.index).astype(str),
        index=lat.index,
    )


def build_analysis_frame(df_raw, grid_miles=1.5, time_bin_minutes=30):
    """Clean source fields and construct the agreed demand segments."""
    required = [
        "Trip ID", "Trip Date", "Purpose", "Promised Pick-up Time",
        "Pick-up Latitude", "Pick-up Longitude",
        "Drop-off Latitude", "Drop-off Longitude",
    ]
    missing = [column for column in required if column not in df_raw.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    frame = df_raw.loc[:, required].copy()
    frame["Purpose"] = _clean_purpose(frame["Purpose"])
    frame["trip_date"] = pd.to_datetime(frame["Trip Date"], errors="coerce")
    frame["weekday"] = frame["trip_date"].dt.day_name()
    frame["promised_pickup_minutes"] = _time_minutes(frame["Promised Pick-up Time"])
    frame["pickup_time_bin_min"] = (
        np.floor(frame["promised_pickup_minutes"] / time_bin_minutes)
        * time_bin_minutes
    )

    coordinate_columns = [
        "Pick-up Latitude", "Pick-up Longitude",
        "Drop-off Latitude", "Drop-off Longitude",
    ]
    for column in coordinate_columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame.dropna(
        subset=[
            "Trip ID", "trip_date", "weekday", "promised_pickup_minutes",
            *coordinate_columns,
        ]
    ).copy()
    finite = np.isfinite(frame[coordinate_columns].to_numpy(dtype=float)).all(axis=1)
    frame = frame.loc[finite].copy()

    miles_per_degree_latitude = 69.0
    mean_latitude_radians = np.deg2rad(frame["Pick-up Latitude"].mean())
    miles_per_degree_longitude = max(
        69.0 * np.cos(mean_latitude_radians), 1e-6
    )
    latitude_step = grid_miles / miles_per_degree_latitude
    longitude_step = grid_miles / miles_per_degree_longitude
    latitude_origin = min(
        frame["Pick-up Latitude"].min(), frame["Drop-off Latitude"].min()
    )
    longitude_origin = min(
        frame["Pick-up Longitude"].min(), frame["Drop-off Longitude"].min()
    )

    frame["origin_zone"] = _make_grid_zone(
        frame["Pick-up Latitude"], frame["Pick-up Longitude"],
        latitude_origin, longitude_origin, latitude_step, longitude_step,
    )
    frame["destination_zone"] = _make_grid_zone(
        frame["Drop-off Latitude"], frame["Drop-off Longitude"],
        latitude_origin, longitude_origin, latitude_step, longitude_step,
    )
    frame["pickup_time_bin"] = frame["pickup_time_bin_min"].map(
        lambda value: _interval_label(value, time_bin_minutes)
    )
    frame["segment_id"] = (
        frame["Purpose"].astype("string")
        + " | " + frame["origin_zone"].astype("string")
        + " | " + frame["weekday"].astype("string")
    )
    frame["policy_flexible"] = (
        ~frame["Purpose"].isin(RIGID_PURPOSES)
        & frame["Purpose"].ne("Missing / Unknown")
    )
    metadata = {
        "grid_miles": grid_miles,
        "time_bin_minutes": time_bin_minutes,
        "latitude_origin": latitude_origin,
        "longitude_origin": longitude_origin,
        "latitude_step": latitude_step,
        "longitude_step": longitude_step,
    }
    return frame, metadata


def calculate_temporal_flexibility(
    frame,
    shift_horizon_minutes=120,
    min_segment_trips=10,
    min_segment_service_days=3,
):
    """Calculate the peak-based distance-weighted temporal index."""
    keys = ["segment_id", "Purpose", "origin_zone", "weekday"]
    distribution = (
        frame.groupby(
            keys + ["pickup_time_bin_min", "pickup_time_bin"], observed=True
        )
        .agg(
            time_bin_trips=("Trip ID", "count"),
            time_bin_service_days=("trip_date", "nunique"),
        )
        .reset_index()
    )
    segment_totals = (
        frame.groupby(keys, observed=True)
        .agg(
            trips=("Trip ID", "count"),
            service_days=("trip_date", "nunique"),
            policy_flexible=("policy_flexible", "first"),
        )
        .reset_index()
    )
    distribution = distribution.merge(segment_totals, on=keys, how="left")
    distribution["p_time"] = distribution["time_bin_trips"] / distribution["trips"]
    distribution["p_s_t"] = distribution["p_time"]

    peak = (
        distribution.sort_values(
            ["segment_id", "time_bin_trips", "pickup_time_bin_min"],
            ascending=[True, False, True],
        )
        .drop_duplicates("segment_id")
        .loc[:, [
            "segment_id", "pickup_time_bin_min", "pickup_time_bin", "p_time"
        ]]
        .rename(columns={
            "pickup_time_bin_min": "peak_time_bin_min",
            "pickup_time_bin": "peak_time_bin",
            "p_time": "peak_time_share",
        })
    )
    distribution = distribution.merge(peak, on="segment_id", how="left")
    signed = distribution["pickup_time_bin_min"] - distribution["peak_time_bin_min"]
    distribution["signed_shift_minutes"] = np.where(
        signed > 720, signed - 1440,
        np.where(signed < -720, signed + 1440, signed),
    )
    distribution["absolute_shift_minutes"] = distribution[
        "signed_shift_minutes"
    ].abs()
    distribution["is_peak_time"] = distribution["absolute_shift_minutes"].eq(0)
    distribution["is_alternative_time"] = ~distribution["is_peak_time"]
    distribution["capped_shift_minutes"] = distribution[
        "absolute_shift_minutes"
    ].clip(upper=shift_horizon_minutes)
    distribution["expected_shift_contribution_minutes"] = (
        distribution["p_time"] * distribution["absolute_shift_minutes"]
    )
    distribution["temporal_index_contribution"] = (
        distribution["p_time"]
        * distribution["capped_shift_minutes"]
        / shift_horizon_minutes
    )

    def summarize(group):
        peak_share = float(group["peak_time_share"].iloc[0])
        alternative_share = 1.0 - peak_share
        alternatives = group.loc[group["is_alternative_time"]]
        weighted_shift = float(
            alternatives["expected_shift_contribution_minutes"].sum()
        )
        average_shift = (
            weighted_shift / alternative_share if alternative_share > 0 else np.nan
        )
        return pd.Series({
            "peak_time_bin": group["peak_time_bin"].iloc[0],
            "peak_time_share": peak_share,
            "active_time_bins": group["pickup_time_bin"].nunique(),
            "alternative_time_share_raw": alternative_share,
            "average_alternative_shift_minutes_raw": average_shift,
            "expected_shift_potential_minutes_raw": weighted_shift,
            "temporal_flexibility_index_raw": float(
                alternatives["temporal_index_contribution"].sum()
            ),
        })

    temporal_metrics = (
        distribution.groupby("segment_id", observed=True, group_keys=False)
        .apply(summarize)
        .reset_index()
    )
    summary = segment_totals.merge(temporal_metrics, on="segment_id", how="left")
    summary["meets_reliability_filter"] = (
        summary["trips"].ge(min_segment_trips)
        & summary["service_days"].ge(min_segment_service_days)
    )
    summary["temporal_eligible"] = (
        summary["policy_flexible"] & summary["meets_reliability_filter"]
    )
    final_to_raw = {
        "alternative_time_share": "alternative_time_share_raw",
        "average_alternative_shift_minutes": "average_alternative_shift_minutes_raw",
        "expected_shift_potential_minutes": "expected_shift_potential_minutes_raw",
        "temporal_flexibility_index": "temporal_flexibility_index_raw",
    }
    for final_column, raw_column in final_to_raw.items():
        summary[final_column] = summary[raw_column].where(
            summary["temporal_eligible"]
        )
    summary["A_s"] = summary["alternative_time_share"]
    summary["D_s_minutes"] = summary["average_alternative_shift_minutes"]
    summary["T_s"] = summary["temporal_flexibility_index"]

    distribution = distribution.merge(
        summary.loc[:, [
            "segment_id", "temporal_eligible", "meets_reliability_filter",
            *final_to_raw.keys(), "A_s", "D_s_minutes", "T_s",
        ]],
        on="segment_id",
        how="left",
    )
    candidate_pool = distribution.loc[
        distribution["temporal_eligible"] & distribution["is_alternative_time"]
    ].copy()
    candidate_pool["alternative_rank"] = (
        candidate_pool.groupby("segment_id")["time_bin_trips"]
        .rank(method="first", ascending=False)
        .astype(int)
    )
    candidate_pool = candidate_pool.sort_values(
        ["segment_id", "alternative_rank", "absolute_shift_minutes"]
    )
    return summary, distribution, candidate_pool


def calculate_geographical_flexibility(frame, segment_summary):
    """Calculate Gini-Simpson destination diversity for allowable purposes."""
    keys = ["segment_id", "Purpose", "origin_zone", "weekday"]
    eligible_frame = frame.loc[frame["policy_flexible"]].copy()
    candidate_pool = (
        eligible_frame.groupby(keys + ["destination_zone"], observed=True)
        .agg(
            destination_visits=("Trip ID", "count"),
            destination_service_days=("trip_date", "nunique"),
        )
        .reset_index()
    )
    totals = candidate_pool.groupby("segment_id")[
        "destination_visits"
    ].transform("sum")
    candidate_pool["destination_share"] = (
        candidate_pool["destination_visits"] / totals
    )
    candidate_pool["p_s_d"] = candidate_pool["destination_share"]
    candidate_pool = candidate_pool.sort_values(
        ["segment_id", "destination_visits", "destination_zone"],
        ascending=[True, False, True],
    )
    candidate_pool["destination_rank"] = (
        candidate_pool.groupby("segment_id").cumcount() + 1
    )
    candidate_pool["is_dominant_destination"] = candidate_pool[
        "destination_rank"
    ].eq(1)

    geo_metrics = (
        candidate_pool.groupby("segment_id", observed=True)
        .agg(
            active_destination_zones=("destination_zone", "nunique"),
            sum_squared_destination_shares=(
                "destination_share", lambda values: float(np.square(values).sum())
            ),
        )
        .reset_index()
    )
    geo_metrics["alternative_destination_count_raw"] = (
        geo_metrics["active_destination_zones"] - 1
    ).clip(lower=0)
    geo_metrics["effective_destination_count_raw"] = (
        1.0 / geo_metrics["sum_squared_destination_shares"]
    )
    geo_metrics["geographical_flexibility_index_raw"] = (
        1.0 - geo_metrics["sum_squared_destination_shares"]
    )
    dominant = (
        candidate_pool.loc[candidate_pool["is_dominant_destination"], [
            "segment_id", "destination_zone", "destination_visits",
            "destination_share",
        ]]
        .rename(columns={
            "destination_zone": "dominant_destination_zone",
            "destination_visits": "dominant_destination_visits",
            "destination_share": "dominant_destination_share",
        })
    )
    geo_metrics = geo_metrics.merge(dominant, on="segment_id", how="left")
    summary = segment_summary.merge(geo_metrics, on="segment_id", how="left")
    summary["geographical_eligible"] = (
        summary["policy_flexible"] & summary["meets_reliability_filter"]
    )
    final_to_raw = {
        "alternative_destination_count": "alternative_destination_count_raw",
        "effective_destination_count": "effective_destination_count_raw",
        "geographical_flexibility_index": "geographical_flexibility_index_raw",
        "dominant_destination_zone_final": "dominant_destination_zone",
        "dominant_destination_visits_final": "dominant_destination_visits",
        "dominant_destination_share_final": "dominant_destination_share",
    }
    for final_column, raw_column in final_to_raw.items():
        summary[final_column] = summary[raw_column].where(
            summary["geographical_eligible"]
        )
    summary["G_s"] = summary["geographical_flexibility_index"]
    return summary, candidate_pool


def run_two_index_analysis(
    df_raw,
    grid_miles=1.5,
    time_bin_minutes=30,
    shift_horizon_minutes=120,
    min_segment_trips=10,
    min_segment_service_days=3,
):
    frame, metadata = build_analysis_frame(
        df_raw, grid_miles=grid_miles, time_bin_minutes=time_bin_minutes
    )
    temporal_summary, time_distribution, temporal_candidates = (
        calculate_temporal_flexibility(
            frame,
            shift_horizon_minutes=shift_horizon_minutes,
            min_segment_trips=min_segment_trips,
            min_segment_service_days=min_segment_service_days,
        )
    )
    segment_summary, geographical_candidates = (
        calculate_geographical_flexibility(frame, temporal_summary)
    )
    parameters = {
        "grid_miles": grid_miles,
        "time_bin_minutes": time_bin_minutes,
        "shift_horizon_minutes": shift_horizon_minutes,
        "min_segment_trips": min_segment_trips,
        "min_segment_service_days": min_segment_service_days,
        "rigid_purposes": sorted(RIGID_PURPOSES),
    }
    return {
        "analysis_frame": frame,
        "segment_summary": segment_summary,
        "time_distribution": time_distribution,
        "temporal_candidate_pool": temporal_candidates,
        "geographical_candidate_pool": geographical_candidates,
        "metadata": metadata,
        "parameters": parameters,
    }


In [4]:
results = run_two_index_analysis(
    df_raw,
    grid_miles=GRID_MILES,
    time_bin_minutes=TIME_BIN_MINUTES,
    shift_horizon_minutes=SHIFT_HORIZON_MINUTES,
    min_segment_trips=MIN_SEGMENT_TRIPS,
    min_segment_service_days=MIN_SEGMENT_SERVICE_DAYS,
)

analysis_frame = results["analysis_frame"]
segment_summary = results["segment_summary"]
time_distribution = results["time_distribution"]
temporal_candidate_pool = results["temporal_candidate_pool"]
geographical_candidate_pool = results["geographical_candidate_pool"]

print(f"Usable trips: {len(analysis_frame):,}")
print(f"Demand segments: {len(segment_summary):,}")
print(f"Reliable segments: {segment_summary['meets_reliability_filter'].sum():,}")
print(f"Temporal scores: {segment_summary['temporal_flexibility_index'].notna().sum():,}")
print(f"Geographical scores: {segment_summary['geographical_flexibility_index'].notna().sum():,}")
print(f"Temporal candidate rows: {len(temporal_candidate_pool):,}")
print(f"Geographical candidate rows: {len(geographical_candidate_pool):,}")


/tmp/ipykernel_1898413/4019214791.py:227: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize)


Usable trips: 121,281
Demand segments: 3,400
Reliable segments: 1,527
Temporal scores: 745
Geographical scores: 745
Temporal candidate rows: 2,091
Geographical candidate rows: 3,060


## Eligibility and reliability

A segment must have at least 10 trips and appear on at least 3 distinct calendar service dates. Policy eligibility and statistical reliability are separate:

- Policy-rigid segment: both indices are NA by definition.
- Eligible but unreliable segment: both published indices are NA because the history is too sparse.
- Eligible and reliable segment: both indices are calculated.


In [5]:
policy_summary = (
    segment_summary.groupby(["Purpose", "policy_flexible"], observed=True)
    .agg(
        segments=("segment_id", "count"),
        trips=("trips", "sum"),
        reliable_segments=("meets_reliability_filter", "sum"),
        temporal_scores=("temporal_flexibility_index", "count"),
        geographical_scores=("geographical_flexibility_index", "count"),
    )
    .reset_index()
    .sort_values("trips", ascending=False)
)
display(policy_summary)

assert not temporal_candidate_pool["Purpose"].isin(RIGID_PURPOSES).any()
assert not geographical_candidate_pool["Purpose"].isin(RIGID_PURPOSES).any()
print("Check passed: rigid purposes are absent from both actionable candidate pools.")


,Purpose,policy_flexible,segments,trips,reliable_segments,temporal_scores,geographical_scores
5,Nutrition,True,546,64285,433,433,433
3,Medical,False,1065,17015,453,0,0
2,Employment,False,354,13619,220,0,0
0,Dialysis,True,260,13186,157,157,157
10,Workshop,False,134,5511,71,0,0
6,Personal,True,428,2689,76,76,76
8,Shopping,True,330,2192,56,56,56
1,Education,False,116,1526,38,0,0
7,Recreation,True,135,1001,23,23,23
4,Missing / Unknown,False,25,245,0,0,0


Check passed: rigid purposes are absent from both actionable candidate pools.


## Temporal results

The summary contains $p_{s,t_s^*}$, $A_s$, $D_s$, expected shift-potential minutes, and $T_s$. The candidate table below shows how the score is constructed from individual alternative time bins.


In [6]:
temporal_columns = [
    "Purpose", "origin_zone", "weekday", "trips", "service_days",
    "peak_time_bin", "peak_time_share", "active_time_bins",
    "A_s", "D_s_minutes", "expected_shift_potential_minutes", "T_s",
]
top_temporal = (
    segment_summary.loc[segment_summary["temporal_eligible"]]
    .nlargest(15, "temporal_flexibility_index")
)
display(
    top_temporal[temporal_columns].style.format({
        "peak_time_share": "{:.1%}",
        "A_s": "{:.1%}",
        "D_s_minutes": "{:.1f}",
        "expected_shift_potential_minutes": "{:.1f}",
        "T_s": "{:.4f}",
    })
)


,Purpose,origin_zone,weekday,trips,service_days,peak_time_bin,peak_time_share,active_time_bins,A_s,D_s_minutes,expected_shift_potential_minutes,T_s
2480,Personal,r22_c16,Tuesday,20,3,10:00-10:30,15.0%,10,85.0%,232.9,198.0,0.8500
2396,Personal,r14_c24,Friday,24,10,10:30-11:00,12.5%,13,87.5%,181.4,158.8,0.7604
2461,Personal,r21_c17,Friday,10,7,09:00-09:30,20.0%,7,80.0%,206.2,165.0,0.7500
2620,Personal,r29_c17,Thursday,19,6,13:00-13:30,15.8%,10,84.2%,131.2,110.5,0.7368
2496,Personal,r23_c16,Tuesday,10,4,11:30-12:00,30.0%,5,70.0%,188.6,132.0,0.7000
2618,Personal,r29_c17,Friday,25,6,10:00-10:30,20.0%,9,80.0%,205.5,164.4,0.7000
2929,Shopping,r10_c14,Friday,13,7,12:00-12:30,30.8%,5,69.2%,140.0,96.9,0.6731
154,Dialysis,r29_c17,Wednesday,82,49,15:30-16:00,29.3%,12,70.7%,219.3,155.1,0.6616
2504,Personal,r24_c11,Friday,10,4,13:30-14:00,20.0%,6,80.0%,105.0,84.0,0.6500
2397,Personal,r14_c24,Monday,12,6,09:00-09:30,16.7%,7,83.3%,120.0,100.0,0.6458


In [7]:
top_temporal_ids = top_temporal.head(5)["segment_id"]
temporal_candidate_examples = (
    temporal_candidate_pool.loc[
        temporal_candidate_pool["segment_id"].isin(top_temporal_ids)
    ]
    .sort_values(["temporal_flexibility_index", "segment_id", "alternative_rank"],
                 ascending=[False, True, True])
)
display(
    temporal_candidate_examples[[
        "Purpose", "origin_zone", "weekday", "peak_time_bin",
        "pickup_time_bin", "time_bin_trips", "p_s_t",
        "signed_shift_minutes", "absolute_shift_minutes",
        "A_s", "D_s_minutes", "temporal_index_contribution", "T_s",
    ]].style.format({
        "p_s_t": "{:.1%}",
        "A_s": "{:.1%}",
        "D_s_minutes": "{:.1f}",
        "temporal_index_contribution": "{:.4f}",
        "T_s": "{:.4f}",
    })
)


,Purpose,origin_zone,weekday,peak_time_bin,pickup_time_bin,time_bin_trips,p_s_t,signed_shift_minutes,absolute_shift_minutes,A_s,D_s_minutes,temporal_index_contribution,T_s
9329,Personal,r22_c16,Tuesday,10:00-10:30,13:00-13:30,3,15.0%,180.000000,180.000000,85.0%,232.9,0.1500,0.8500
9327,Personal,r22_c16,Tuesday,10:00-10:30,12:00-12:30,2,10.0%,120.000000,120.000000,85.0%,232.9,0.1000,0.8500
9328,Personal,r22_c16,Tuesday,10:00-10:30,12:30-13:00,2,10.0%,150.000000,150.000000,85.0%,232.9,0.1000,0.8500
9331,Personal,r22_c16,Tuesday,10:00-10:30,14:00-14:30,2,10.0%,240.000000,240.000000,85.0%,232.9,0.1000,0.8500
9332,Personal,r22_c16,Tuesday,10:00-10:30,14:30-15:00,2,10.0%,270.000000,270.000000,85.0%,232.9,0.1000,0.8500
9333,Personal,r22_c16,Tuesday,10:00-10:30,15:00-15:30,2,10.0%,300.000000,300.000000,85.0%,232.9,0.1000,0.8500
9335,Personal,r22_c16,Tuesday,10:00-10:30,16:00-16:30,2,10.0%,360.000000,360.000000,85.0%,232.9,0.1000,0.8500
9330,Personal,r22_c16,Tuesday,10:00-10:30,13:30-14:00,1,5.0%,210.000000,210.000000,85.0%,232.9,0.0500,0.8500
9334,Personal,r22_c16,Tuesday,10:00-10:30,15:30-16:00,1,5.0%,330.000000,330.000000,85.0%,232.9,0.0500,0.8500
9029,Personal,r14_c24,Friday,10:30-11:00,13:00-13:30,3,12.5%,150.000000,150.000000,87.5%,181.4,0.1250,0.7604


## Geographical results

The summary reports the Gini-Simpson index, effective number of destinations, and dominant destination. The candidate table explicitly ranks locations by visits, so the highest-visit destination remains visible.


In [8]:
geographical_columns = [
    "Purpose", "origin_zone", "weekday", "trips", "service_days",
    "active_destination_zones", "alternative_destination_count",
    "effective_destination_count", "dominant_destination_zone_final",
    "dominant_destination_visits_final", "dominant_destination_share_final",
    "G_s",
]
top_geographical = (
    segment_summary.loc[segment_summary["geographical_eligible"]]
    .nlargest(15, "geographical_flexibility_index")
)
display(
    top_geographical[geographical_columns].style.format({
        "effective_destination_count": "{:.2f}",
        "dominant_destination_share_final": "{:.1%}",
        "G_s": "{:.4f}",
    })
)


,Purpose,origin_zone,weekday,trips,service_days,active_destination_zones,alternative_destination_count,effective_destination_count,dominant_destination_zone_final,dominant_destination_visits_final,dominant_destination_share_final,G_s
2865,Recreation,r29_c15,Thursday,44,6,14.000000,13.000000,10.19,r33_c16,7.000000,15.9%,0.9019
1909,Nutrition,r16_c16,Monday,320,52,9.000000,8.000000,6.94,r16_c14,60.000000,18.8%,0.8559
2723,Personal,r34_c18,Thursday,26,14,8.000000,7.000000,6.63,r34_c24,6.000000,23.1%,0.8491
1997,Nutrition,r23_c16,Thursday,822,52,11.000000,10.000000,6.61,r24_c17,220.000000,26.8%,0.8486
1912,Nutrition,r16_c16,Wednesday,463,52,10.000000,9.000000,6.52,r16_c16,129.000000,27.9%,0.8466
2398,Personal,r14_c24,Thursday,25,10,9.000000,8.000000,6.44,r13_c24,6.000000,24.0%,0.8448
1995,Nutrition,r23_c16,Friday,757,53,8.000000,7.000000,5.94,r24_c17,203.000000,26.8%,0.8317
1996,Nutrition,r23_c16,Monday,693,52,8.000000,7.000000,5.93,r24_c17,187.000000,27.0%,0.8314
3233,Shopping,r35_c23,Thursday,30,14,9.000000,8.000000,5.84,r34_c18,8.000000,26.7%,0.8289
3231,Shopping,r35_c23,Friday,18,11,7.000000,6.000000,5.79,r28_c14,4.000000,22.2%,0.8272


In [9]:
top_geo_ids = top_geographical.head(5)["segment_id"]
geographical_candidate_examples = (
    geographical_candidate_pool.loc[
        geographical_candidate_pool["segment_id"].isin(top_geo_ids)
        & geographical_candidate_pool["destination_rank"].le(10)
    ]
    .sort_values(["segment_id", "destination_rank"])
)
display(
    geographical_candidate_examples[[
        "Purpose", "origin_zone", "weekday", "destination_zone",
        "destination_visits", "p_s_d", "destination_rank",
        "is_dominant_destination",
    ]].style.format({"p_s_d": "{:.1%}"})
)


,Purpose,origin_zone,weekday,destination_zone,destination_visits,p_s_d,destination_rank,is_dominant_destination
569,Nutrition,r16_c16,Monday,r16_c14,60,18.8%,1,True
573,Nutrition,r16_c16,Monday,r17_c17,55,17.2%,2,False
574,Nutrition,r16_c16,Monday,r18_c16,52,16.2%,3,False
575,Nutrition,r16_c16,Monday,r20_c20,50,15.6%,4,False
570,Nutrition,r16_c16,Monday,r16_c15,37,11.6%,5,False
567,Nutrition,r16_c16,Monday,r14_c16,32,10.0%,6,False
568,Nutrition,r16_c16,Monday,r15_c15,21,6.6%,7,False
572,Nutrition,r16_c16,Monday,r17_c16,9,2.8%,8,False
571,Nutrition,r16_c16,Monday,r17_c15,4,1.2%,9,False
594,Nutrition,r16_c16,Wednesday,r16_c16,129,27.9%,1,True


## Export tables and generate the results report

The final cell exports the segment summary and both actionable candidate pools, then creates a Markdown report directly from the calculated results.


In [10]:
SEGMENT_CSV = PROJECT_DIR / "two_index_segment_summary.csv"
TEMPORAL_CSV = PROJECT_DIR / "temporal_flexibility_candidate_pool.csv"
GEOGRAPHICAL_CSV = PROJECT_DIR / "geographical_flexibility_candidate_pool.csv"
REPORT_PATH = PROJECT_DIR / "two_index_flexibility_results.md"

segment_summary.to_csv(SEGMENT_CSV, index=False)
temporal_candidate_pool.to_csv(TEMPORAL_CSV, index=False)
geographical_candidate_pool.to_csv(GEOGRAPHICAL_CSV, index=False)

def index_statistics(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    return {
        "n": len(values),
        "mean": values.mean(),
        "q25": values.quantile(0.25),
        "median": values.median(),
        "q75": values.quantile(0.75),
        "max": values.max(),
    }

def md_table(data, formats=None):
    table = data.copy()
    for column, formatter in (formats or {}).items():
        if column in table:
            table[column] = table[column].map(
                lambda value: "" if pd.isna(value) else formatter(value)
            )
    table = table.fillna("").astype(str)
    headers = [str(column).replace("|", "\\|") for column in table.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in table.itertuples(index=False, name=None):
        values = [str(value).replace("|", "\\|").replace("\n", " ") for value in row]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)

temporal_stats = index_statistics(segment_summary["temporal_flexibility_index"])
geo_stats = index_statistics(segment_summary["geographical_flexibility_index"])
distribution_table = pd.DataFrame([
    {"index": "Temporal", **temporal_stats},
    {"index": "Geographical", **geo_stats},
])

top_temporal_report = top_temporal[temporal_columns].head(10)
top_geo_report = top_geographical[geographical_columns].head(10)

temporal_example_report = temporal_candidate_examples[[
    "Purpose", "origin_zone", "weekday", "peak_time_bin",
    "pickup_time_bin", "time_bin_trips", "p_s_t",
    "signed_shift_minutes", "temporal_index_contribution",
]].head(20)

geo_example_report = geographical_candidate_examples[[
    "Purpose", "origin_zone", "weekday", "destination_zone",
    "destination_visits", "p_s_d", "destination_rank",
    "is_dominant_destination",
]].head(30)

top_t = top_temporal.iloc[0]
top_g = top_geographical.iloc[0]

report_lines = [
    "# ClassTran temporal and geographical flexibility results",
    "",
    f"Source workbook: {DATA_PATH.name}",
    "",
    "## Demand segment and policy",
    "",
    f"Demand segment = Purpose + {GRID_MILES:g}-mile origin grid zone + weekday.",
    "",
    "Workshop, Employment, Education, and Medical are treated as rigid. "
    "Both indices are NA for those purposes, and their candidate pools are empty.",
    "",
    "A reliable segment has at least 10 trips and at least 3 distinct service dates.",
    "",
    "## Temporal index construction",
    "",
    "Promised Pick-up Time is grouped into 30-minute half-open bins. "
    "For segment s and time bin t, p(s,t) is the bin's trip share and t* is the peak bin.",
    "",
    "$$A_s=1-p_{s,t_s^*}$$",
    "",
    "### A_s: alternative-time share",
    "",
    "A_s is the proportion of trips in segment s historically observed outside "
    "the peak 30-minute bin. It ranges from 0 to 1 and is usually displayed as "
    "a percentage. A_s = 0 means no alternative-time evidence; A_s = 0.40 means "
    "40% of trips occurred outside the peak. It measures how common alternatives "
    "are, not how far away they are, and it is not the guaranteed share of current "
    "peak trips that can be moved.",
    "",
    "$$D_s=\\frac{\\sum_{t\\ne t_s^*}p_{s,t}|t-t_s^*|}{A_s}$$",
    "",
    "### D_s: average alternative shift",
    "",
    "D_s is the weighted average absolute distance, in minutes, from the peak "
    "among non-peak observations. A value of 30 means alternatives are on average "
    "30 minutes from the peak. D_s is NA when A_s = 0. Because it is absolute, it "
    "does not show direction; signed_shift_minutes in the candidate table identifies "
    "earlier (negative) and later (positive) alternatives.",
    "",
    "Using H = 120 minutes:",
    "",
    "$$T_s=\\sum_{t\\ne t_s^*}p_{s,t}"
    "\\min\\left(\\frac{|t-t_s^*|}{H},1\\right)$$",
    "",
    "### T_s: temporal flexibility index",
    "",
    "T_s combines alternative-time prevalence and distance. It ranges from 0 to 1 "
    "and has no unit. Each distance is normalized by H = 120 minutes and capped at "
    "one, so rare extreme times cannot dominate. T_s = 0 means no non-peak time was "
    "observed; higher values mean alternatives are more prevalent, farther from the "
    "peak, or both.",
    "",
    "When all alternative distances are at most H, T_s = A_s(D_s/H). Therefore, "
    "neither high A_s nor high D_s alone guarantees high T_s. This is observed "
    "segment-level shift potential, not evidence of individual rider consent.",
    "",
    "## Geographical index construction",
    "",
    f"For allowable purposes, candidates are observed {GRID_MILES:g}-mile destination zones "
    "from the same demand segment. For destination share p(s,d):",
    "",
    "### p(s,d): destination visit share",
    "",
    "p(s,d) = n(s,d)/N(s), where n(s,d) is the number of visits to destination "
    "zone d and N(s) is the segment's total trips. It ranges from 0 to 1, all "
    "shares within a segment sum to 1, and the destination with the largest share "
    "is the dominant destination.",
    "",
    "$$G_s=1-\\sum_d p_{s,d}^2$$",
    "",
    "### G_s: geographical flexibility index",
    "",
    "G_s is the Gini-Simpson diversity index. It ranges from 0 to less than 1 "
    "and incorporates both the number of destinations and the balance of their "
    "visit shares. G_s = 0 means every trip uses one destination. It can also be "
    "interpreted as the probability that two randomly selected trips from the "
    "segment have different destination zones. With equal shares, one, two, five, "
    "and ten destinations produce scores of 0, 0.50, 0.80, and 0.90.",
    "",
    "$$N_{effective}=\\frac{1}{\\sum_d p_{s,d}^2}$$",
    "",
    "### N_effective: effective destination count",
    "",
    "N_effective is the number of equally used destinations that would have the "
    "same diversity as the observed distribution. Its minimum is 1. For example, "
    "N_effective = 3 means the distribution has the same diversity as three equally "
    "used zones, even if more zones were observed. It is related to G_s by "
    "G_s = 1 - 1/N_effective.",
    "",
    "### Why the ranked candidate table is also necessary",
    "",
    "The indices summarize diversity but do not identify locations. The candidate "
    "table reports destination zone, visits, p(s,d), rank, and a dominant-destination "
    "flag. Rank 1 is the highest-visit destination; lower ranks are observed alternatives. "
    "Rigid purposes have empty candidate pools and an NA geographical index.",
    "",
    "## Data coverage",
    "",
    f"- Loaded source records: {len(df_raw):,}",
    f"- Usable records: {len(analysis_frame):,}",
    f"- Demand segments: {len(segment_summary):,}",
    f"- Reliable segments: {int(segment_summary['meets_reliability_filter'].sum()):,}",
    f"- Published temporal scores: "
    f"{int(segment_summary['temporal_flexibility_index'].notna().sum()):,}",
    f"- Published geographical scores: "
    f"{int(segment_summary['geographical_flexibility_index'].notna().sum()):,}",
    f"- Temporal alternative rows: {len(temporal_candidate_pool):,}",
    f"- Geographical candidate rows: {len(geographical_candidate_pool):,}",
    "",
    "### Purpose eligibility and reliability",
    "",
    md_table(policy_summary),
    "",
    "## Index distributions",
    "",
    md_table(distribution_table, {
        "mean": lambda value: f"{value:.4f}",
        "q25": lambda value: f"{value:.4f}",
        "median": lambda value: f"{value:.4f}",
        "q75": lambda value: f"{value:.4f}",
        "max": lambda value: f"{value:.4f}",
    }),
    "",
    "## Highest observed temporal flexibility segments",
    "",
    md_table(top_temporal_report, {
        "peak_time_share": lambda value: f"{value:.1%}",
        "A_s": lambda value: f"{value:.1%}",
        "D_s_minutes": lambda value: f"{value:.1f}",
        "expected_shift_potential_minutes": lambda value: f"{value:.1f}",
        "T_s": lambda value: f"{value:.4f}",
    }),
    "",
    "### Example temporal alternatives",
    "",
    md_table(temporal_example_report, {
        "p_s_t": lambda value: f"{value:.1%}",
        "signed_shift_minutes": lambda value: f"{value:+.0f}",
        "temporal_index_contribution": lambda value: f"{value:.4f}",
    }),
    "",
    "## Highest observed geographical flexibility segments",
    "",
    md_table(top_geo_report, {
        "effective_destination_count": lambda value: f"{value:.2f}",
        "dominant_destination_share_final": lambda value: f"{value:.1%}",
        "G_s": lambda value: f"{value:.4f}",
    }),
    "",
    "### Example ranked destination candidates",
    "",
    md_table(geo_example_report, {
        "p_s_d": lambda value: f"{value:.1%}",
    }),
    "",
    "## Observations",
    "",
    f"- The median temporal index among reliable eligible segments is "
    f"{temporal_stats['median']:.4f}; the middle 50% ranges from "
    f"{temporal_stats['q25']:.4f} to {temporal_stats['q75']:.4f}.",
    f"- The highest temporal score is {top_t['temporal_flexibility_index']:.4f} "
    f"for {top_t['Purpose']} in {top_t['origin_zone']} on {top_t['weekday']}. "
    f"Its peak is {top_t['peak_time_bin']}, its alternative-time share is "
    f"{top_t['alternative_time_share']:.1%}, and its conditional average "
    f"alternative shift is {top_t['average_alternative_shift_minutes']:.1f} minutes.",
    f"- The median geographical index is {geo_stats['median']:.4f}; the middle "
    f"50% ranges from {geo_stats['q25']:.4f} to {geo_stats['q75']:.4f}.",
    f"- The highest geographical score is "
    f"{top_g['geographical_flexibility_index']:.4f} for {top_g['Purpose']} in "
    f"{top_g['origin_zone']} on {top_g['weekday']}, with "
    f"{int(top_g['active_destination_zones'])} observed destination zones and "
    f"{top_g['effective_destination_count']:.2f} effective destinations.",
    "",
    "## Demand-management use",
    "",
    "Use T to screen segments with meaningful non-peak time alternatives, then "
    "use the temporal candidate table to select earlier or later bins and see "
    "their historical shares. Use G to screen destination-diverse segments, then "
    "use destination rank, visits, and share to identify the dominant and alternative locations.",
    "",
    "These are observational planning measures. They do not establish that an "
    "individual trip can be shifted without rider consent, service constraints, "
    "capacity checks, and purpose-specific operational review.",
    "",
    "## Grid-size sensitivity",
    "",
    "The current main results above use 1.5-mile origin and destination grid zones. "
    "I also checked 0.5-, 1.0-, 1.5-, and 2.0-mile grid sizes using the same "
    "demand-segment definition, 30-minute time bins, minimum 10 trips, and minimum "
    "3 service days.",
    "",
    "| Grid size | Demand segments | Reliable segments | Reliable share | Published scores | Median segment trips | Median reliable trips | Median T | Median G | Geographical candidate rows |",
    "|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
    "| 0.5 mi | 4,813 | 1,734 | 36.0% | 906 | 5 | 39 | 0.0876 | 0.0000 | 3,594 |",
    "| 1.0 mi | 4,016 | 1,643 | 40.9% | 820 | 7 | 40 | 0.1072 | 0.0000 | 3,297 |",
    "| 1.5 mi | 3,400 | 1,527 | 44.9% | 745 | 8 | 38 | 0.1237 | 0.0000 | 3,060 |",
    "| 2.0 mi | 2,845 | 1,390 | 48.9% | 673 | 10 | 45 | 0.1439 | 0.0411 | 2,785 |",
    "",
    "The finer grids create more demand segments and more published scores, but "
    "the median segment becomes smaller. The 0.5-mile version has a median of "
    "only 5 trips per segment, so it gives more spatial detail but weaker "
    "segment-level stability. The 2.0-mile version has fewer published segments, "
    "but each segment is denser and the geographical index becomes less often "
    "zero. The 1.5-mile grid is the selected middle case: it is less sparse than "
    "0.5 or 1.0 miles, but still preserves more spatial detail than 2.0 miles.",
    "",
]

REPORT_PATH.write_text("\n".join(report_lines))

print(f"Saved: {SEGMENT_CSV.name}")
print(f"Saved: {TEMPORAL_CSV.name}")
print(f"Saved: {GEOGRAPHICAL_CSV.name}")
print(f"Saved: {REPORT_PATH.name}")
display(Markdown(REPORT_PATH.read_text()))


Saved: two_index_segment_summary.csv
Saved: temporal_flexibility_candidate_pool.csv
Saved: geographical_flexibility_candidate_pool.csv
Saved: two_index_flexibility_results.md


# ClassTran temporal and geographical flexibility results

Source workbook: Ecolane Reservation and Trip Data July 2022 - June 2023.xlsx

## Demand segment and policy

Demand segment = Purpose + 1.5-mile origin grid zone + weekday.

Workshop, Employment, Education, and Medical are treated as rigid. Both indices are NA for those purposes, and their candidate pools are empty.

A reliable segment has at least 10 trips and at least 3 distinct service dates.

## Temporal index construction

Promised Pick-up Time is grouped into 30-minute half-open bins. For segment s and time bin t, p(s,t) is the bin's trip share and t* is the peak bin.

$$A_s=1-p_{s,t_s^*}$$

### A_s: alternative-time share

A_s is the proportion of trips in segment s historically observed outside the peak 30-minute bin. It ranges from 0 to 1 and is usually displayed as a percentage. A_s = 0 means no alternative-time evidence; A_s = 0.40 means 40% of trips occurred outside the peak. It measures how common alternatives are, not how far away they are, and it is not the guaranteed share of current peak trips that can be moved.

$$D_s=\frac{\sum_{t\ne t_s^*}p_{s,t}|t-t_s^*|}{A_s}$$

### D_s: average alternative shift

D_s is the weighted average absolute distance, in minutes, from the peak among non-peak observations. A value of 30 means alternatives are on average 30 minutes from the peak. D_s is NA when A_s = 0. Because it is absolute, it does not show direction; signed_shift_minutes in the candidate table identifies earlier (negative) and later (positive) alternatives.

Using H = 120 minutes:

$$T_s=\sum_{t\ne t_s^*}p_{s,t}\min\left(\frac{|t-t_s^*|}{H},1\right)$$

### T_s: temporal flexibility index

T_s combines alternative-time prevalence and distance. It ranges from 0 to 1 and has no unit. Each distance is normalized by H = 120 minutes and capped at one, so rare extreme times cannot dominate. T_s = 0 means no non-peak time was observed; higher values mean alternatives are more prevalent, farther from the peak, or both.

When all alternative distances are at most H, T_s = A_s(D_s/H). Therefore, neither high A_s nor high D_s alone guarantees high T_s. This is observed segment-level shift potential, not evidence of individual rider consent.

## Geographical index construction

For allowable purposes, candidates are observed 1.5-mile destination zones from the same demand segment. For destination share p(s,d):

### p(s,d): destination visit share

p(s,d) = n(s,d)/N(s), where n(s,d) is the number of visits to destination zone d and N(s) is the segment's total trips. It ranges from 0 to 1, all shares within a segment sum to 1, and the destination with the largest share is the dominant destination.

$$G_s=1-\sum_d p_{s,d}^2$$

### G_s: geographical flexibility index

G_s is the Gini-Simpson diversity index. It ranges from 0 to less than 1 and incorporates both the number of destinations and the balance of their visit shares. G_s = 0 means every trip uses one destination. It can also be interpreted as the probability that two randomly selected trips from the segment have different destination zones. With equal shares, one, two, five, and ten destinations produce scores of 0, 0.50, 0.80, and 0.90.

$$N_{effective}=\frac{1}{\sum_d p_{s,d}^2}$$

### N_effective: effective destination count

N_effective is the number of equally used destinations that would have the same diversity as the observed distribution. Its minimum is 1. For example, N_effective = 3 means the distribution has the same diversity as three equally used zones, even if more zones were observed. It is related to G_s by G_s = 1 - 1/N_effective.

### Why the ranked candidate table is also necessary

The indices summarize diversity but do not identify locations. The candidate table reports destination zone, visits, p(s,d), rank, and a dominant-destination flag. Rank 1 is the highest-visit destination; lower ranks are observed alternatives. Rigid purposes have empty candidate pools and an NA geographical index.

## Data coverage

- Loaded source records: 121,281
- Usable records: 121,281
- Demand segments: 3,400
- Reliable segments: 1,527
- Published temporal scores: 745
- Published geographical scores: 745
- Temporal alternative rows: 2,091
- Geographical candidate rows: 3,060

### Purpose eligibility and reliability

| Purpose | policy_flexible | segments | trips | reliable_segments | temporal_scores | geographical_scores |
| --- | --- | --- | --- | --- | --- | --- |
| Nutrition | True | 546 | 64285 | 433 | 433 | 433 |
| Medical | False | 1065 | 17015 | 453 | 0 | 0 |
| Employment | False | 354 | 13619 | 220 | 0 | 0 |
| Dialysis | True | 260 | 13186 | 157 | 157 | 157 |
| Workshop | False | 134 | 5511 | 71 | 0 | 0 |
| Personal | True | 428 | 2689 | 76 | 76 | 76 |
| Shopping | True | 330 | 2192 | 56 | 56 | 56 |
| Education | False | 116 | 1526 | 38 | 0 | 0 |
| Recreation | True | 135 | 1001 | 23 | 23 | 23 |
| Missing / Unknown | False | 25 | 245 | 0 | 0 | 0 |
| Trolley | True | 7 | 12 | 0 | 0 | 0 |

## Index distributions

| index | n | mean | q25 | median | q75 | max |
| --- | --- | --- | --- | --- | --- | --- |
| Temporal | 745 | 0.1567 | 0.0411 | 0.1237 | 0.2214 | 0.8500 |
| Geographical | 745 | 0.2092 | 0.0000 | 0.0000 | 0.4942 | 0.9019 |

## Highest observed temporal flexibility segments

| Purpose | origin_zone | weekday | trips | service_days | peak_time_bin | peak_time_share | active_time_bins | A_s | D_s_minutes | expected_shift_potential_minutes | T_s |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Personal | r22_c16 | Tuesday | 20 | 3 | 10:00-10:30 | 15.0% | 10 | 85.0% | 232.9 | 198.0 | 0.8500 |
| Personal | r14_c24 | Friday | 24 | 10 | 10:30-11:00 | 12.5% | 13 | 87.5% | 181.4 | 158.8 | 0.7604 |
| Personal | r21_c17 | Friday | 10 | 7 | 09:00-09:30 | 20.0% | 7 | 80.0% | 206.2 | 165.0 | 0.7500 |
| Personal | r29_c17 | Thursday | 19 | 6 | 13:00-13:30 | 15.8% | 10 | 84.2% | 131.2 | 110.5 | 0.7368 |
| Personal | r23_c16 | Tuesday | 10 | 4 | 11:30-12:00 | 30.0% | 5 | 70.0% | 188.6 | 132.0 | 0.7000 |
| Personal | r29_c17 | Friday | 25 | 6 | 10:00-10:30 | 20.0% | 9 | 80.0% | 205.5 | 164.4 | 0.7000 |
| Shopping | r10_c14 | Friday | 13 | 7 | 12:00-12:30 | 30.8% | 5 | 69.2% | 140.0 | 96.9 | 0.6731 |
| Dialysis | r29_c17 | Wednesday | 82 | 49 | 15:30-16:00 | 29.3% | 12 | 70.7% | 219.3 | 155.1 | 0.6616 |
| Personal | r24_c11 | Friday | 10 | 4 | 13:30-14:00 | 20.0% | 6 | 80.0% | 105.0 | 84.0 | 0.6500 |
| Personal | r14_c24 | Monday | 12 | 6 | 09:00-09:30 | 16.7% | 7 | 83.3% | 120.0 | 100.0 | 0.6458 |

### Example temporal alternatives

| Purpose | origin_zone | weekday | peak_time_bin | pickup_time_bin | time_bin_trips | p_s_t | signed_shift_minutes | temporal_index_contribution |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Personal | r22_c16 | Tuesday | 10:00-10:30 | 13:00-13:30 | 3 | 15.0% | +180 | 0.1500 |
| Personal | r22_c16 | Tuesday | 10:00-10:30 | 12:00-12:30 | 2 | 10.0% | +120 | 0.1000 |
| Personal | r22_c16 | Tuesday | 10:00-10:30 | 12:30-13:00 | 2 | 10.0% | +150 | 0.1000 |
| Personal | r22_c16 | Tuesday | 10:00-10:30 | 14:00-14:30 | 2 | 10.0% | +240 | 0.1000 |
| Personal | r22_c16 | Tuesday | 10:00-10:30 | 14:30-15:00 | 2 | 10.0% | +270 | 0.1000 |
| Personal | r22_c16 | Tuesday | 10:00-10:30 | 15:00-15:30 | 2 | 10.0% | +300 | 0.1000 |
| Personal | r22_c16 | Tuesday | 10:00-10:30 | 16:00-16:30 | 2 | 10.0% | +360 | 0.1000 |
| Personal | r22_c16 | Tuesday | 10:00-10:30 | 13:30-14:00 | 1 | 5.0% | +210 | 0.0500 |
| Personal | r22_c16 | Tuesday | 10:00-10:30 | 15:30-16:00 | 1 | 5.0% | +330 | 0.0500 |
| Personal | r14_c24 | Friday | 10:30-11:00 | 13:00-13:30 | 3 | 12.5% | +150 | 0.1250 |
| Personal | r14_c24 | Friday | 10:30-11:00 | 14:00-14:30 | 3 | 12.5% | +210 | 0.1250 |
| Personal | r14_c24 | Friday | 10:30-11:00 | 12:00-12:30 | 2 | 8.3% | +90 | 0.0625 |
| Personal | r14_c24 | Friday | 10:30-11:00 | 13:30-14:00 | 2 | 8.3% | +180 | 0.0833 |
| Personal | r14_c24 | Friday | 10:30-11:00 | 14:30-15:00 | 2 | 8.3% | +240 | 0.0833 |
| Personal | r14_c24 | Friday | 10:30-11:00 | 15:00-15:30 | 2 | 8.3% | +270 | 0.0833 |
| Personal | r14_c24 | Friday | 10:30-11:00 | 16:00-16:30 | 2 | 8.3% | +330 | 0.0833 |
| Personal | r14_c24 | Friday | 10:30-11:00 | 09:00-09:30 | 1 | 4.2% | -90 | 0.0312 |
| Personal | r14_c24 | Friday | 10:30-11:00 | 09:30-10:00 | 1 | 4.2% | -60 | 0.0208 |
| Personal | r14_c24 | Friday | 10:30-11:00 | 10:00-10:30 | 1 | 4.2% | -30 | 0.0104 |
| Personal | r14_c24 | Friday | 10:30-11:00 | 11:00-11:30 | 1 | 4.2% | +30 | 0.0104 |

## Highest observed geographical flexibility segments

| Purpose | origin_zone | weekday | trips | service_days | active_destination_zones | alternative_destination_count | effective_destination_count | dominant_destination_zone_final | dominant_destination_visits_final | dominant_destination_share_final | G_s |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Recreation | r29_c15 | Thursday | 44 | 6 | 14.0 | 13.0 | 10.19 | r33_c16 | 7.0 | 15.9% | 0.9019 |
| Nutrition | r16_c16 | Monday | 320 | 52 | 9.0 | 8.0 | 6.94 | r16_c14 | 60.0 | 18.8% | 0.8559 |
| Personal | r34_c18 | Thursday | 26 | 14 | 8.0 | 7.0 | 6.63 | r34_c24 | 6.0 | 23.1% | 0.8491 |
| Nutrition | r23_c16 | Thursday | 822 | 52 | 11.0 | 10.0 | 6.61 | r24_c17 | 220.0 | 26.8% | 0.8486 |
| Nutrition | r16_c16 | Wednesday | 463 | 52 | 10.0 | 9.0 | 6.52 | r16_c16 | 129.0 | 27.9% | 0.8466 |
| Personal | r14_c24 | Thursday | 25 | 10 | 9.0 | 8.0 | 6.44 | r13_c24 | 6.0 | 24.0% | 0.8448 |
| Nutrition | r23_c16 | Friday | 757 | 53 | 8.0 | 7.0 | 5.94 | r24_c17 | 203.0 | 26.8% | 0.8317 |
| Nutrition | r23_c16 | Monday | 693 | 52 | 8.0 | 7.0 | 5.93 | r24_c17 | 187.0 | 27.0% | 0.8314 |
| Shopping | r35_c23 | Thursday | 30 | 14 | 9.0 | 8.0 | 5.84 | r34_c18 | 8.0 | 26.7% | 0.8289 |
| Shopping | r35_c23 | Friday | 18 | 11 | 7.0 | 6.0 | 5.79 | r28_c14 | 4.0 | 22.2% | 0.8272 |

### Example ranked destination candidates

| Purpose | origin_zone | weekday | destination_zone | destination_visits | p_s_d | destination_rank | is_dominant_destination |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Nutrition | r16_c16 | Monday | r16_c14 | 60 | 18.8% | 1 | True |
| Nutrition | r16_c16 | Monday | r17_c17 | 55 | 17.2% | 2 | False |
| Nutrition | r16_c16 | Monday | r18_c16 | 52 | 16.2% | 3 | False |
| Nutrition | r16_c16 | Monday | r20_c20 | 50 | 15.6% | 4 | False |
| Nutrition | r16_c16 | Monday | r16_c15 | 37 | 11.6% | 5 | False |
| Nutrition | r16_c16 | Monday | r14_c16 | 32 | 10.0% | 6 | False |
| Nutrition | r16_c16 | Monday | r15_c15 | 21 | 6.6% | 7 | False |
| Nutrition | r16_c16 | Monday | r17_c16 | 9 | 2.8% | 8 | False |
| Nutrition | r16_c16 | Monday | r17_c15 | 4 | 1.2% | 9 | False |
| Nutrition | r16_c16 | Wednesday | r16_c16 | 129 | 27.9% | 1 | True |
| Nutrition | r16_c16 | Wednesday | r16_c14 | 66 | 14.3% | 2 | False |
| Nutrition | r16_c16 | Wednesday | r17_c17 | 57 | 12.3% | 3 | False |
| Nutrition | r16_c16 | Wednesday | r18_c16 | 57 | 12.3% | 4 | False |
| Nutrition | r16_c16 | Wednesday | r20_c20 | 52 | 11.2% | 5 | False |
| Nutrition | r16_c16 | Wednesday | r16_c15 | 35 | 7.6% | 6 | False |
| Nutrition | r16_c16 | Wednesday | r14_c16 | 29 | 6.3% | 7 | False |
| Nutrition | r16_c16 | Wednesday | r19_c16 | 22 | 4.8% | 8 | False |
| Nutrition | r16_c16 | Wednesday | r17_c16 | 10 | 2.2% | 9 | False |
| Nutrition | r16_c16 | Wednesday | r17_c15 | 6 | 1.3% | 10 | False |
| Nutrition | r23_c16 | Thursday | r24_c17 | 220 | 26.8% | 1 | True |
| Nutrition | r23_c16 | Thursday | r22_c15 | 161 | 19.6% | 2 | False |
| Nutrition | r23_c16 | Thursday | r23_c16 | 92 | 11.2% | 3 | False |
| Nutrition | r23_c16 | Thursday | r22_c14 | 83 | 10.1% | 4 | False |
| Nutrition | r23_c16 | Thursday | r24_c15 | 56 | 6.8% | 5 | False |
| Nutrition | r23_c16 | Thursday | r23_c17 | 54 | 6.6% | 6 | False |
| Nutrition | r23_c16 | Thursday | r21_c13 | 53 | 6.4% | 7 | False |
| Nutrition | r23_c16 | Thursday | r25_c15 | 48 | 5.8% | 8 | False |
| Nutrition | r23_c16 | Thursday | r21_c14 | 32 | 3.9% | 9 | False |
| Nutrition | r23_c16 | Thursday | r23_c15 | 20 | 2.4% | 10 | False |
| Personal | r34_c18 | Thursday | r34_c24 | 6 | 23.1% | 1 | True |

## Observations

- The median temporal index among reliable eligible segments is 0.1237; the middle 50% ranges from 0.0411 to 0.2214.
- The highest temporal score is 0.8500 for Personal in r22_c16 on Tuesday. Its peak is 10:00-10:30, its alternative-time share is 85.0%, and its conditional average alternative shift is 232.9 minutes.
- The median geographical index is 0.0000; the middle 50% ranges from 0.0000 to 0.4942.
- The highest geographical score is 0.9019 for Recreation in r29_c15 on Thursday, with 14 observed destination zones and 10.19 effective destinations.

## Demand-management use

Use T to screen segments with meaningful non-peak time alternatives, then use the temporal candidate table to select earlier or later bins and see their historical shares. Use G to screen destination-diverse segments, then use destination rank, visits, and share to identify the dominant and alternative locations.

These are observational planning measures. They do not establish that an individual trip can be shifted without rider consent, service constraints, capacity checks, and purpose-specific operational review.

## Grid-size sensitivity

The current main results above use 1.5-mile origin and destination grid zones. I also checked 0.5-, 1.0-, 1.5-, and 2.0-mile grid sizes using the same demand-segment definition, 30-minute time bins, minimum 10 trips, and minimum 3 service days.

| Grid size | Demand segments | Reliable segments | Reliable share | Published scores | Median segment trips | Median reliable trips | Median T | Median G | Geographical candidate rows |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 0.5 mi | 4,813 | 1,734 | 36.0% | 906 | 5 | 39 | 0.0876 | 0.0000 | 3,594 |
| 1.0 mi | 4,016 | 1,643 | 40.9% | 820 | 7 | 40 | 0.1072 | 0.0000 | 3,297 |
| 1.5 mi | 3,400 | 1,527 | 44.9% | 745 | 8 | 38 | 0.1237 | 0.0000 | 3,060 |
| 2.0 mi | 2,845 | 1,390 | 48.9% | 673 | 10 | 45 | 0.1439 | 0.0411 | 2,785 |

The finer grids create more demand segments and more published scores, but the median segment becomes smaller. The 0.5-mile version has a median of only 5 trips per segment, so it gives more spatial detail but weaker segment-level stability. The 2.0-mile version has fewer published segments, but each segment is denser and the geographical index becomes less often zero. The 1.5-mile grid is the selected middle case: it is less sparse than 0.5 or 1.0 miles, but still preserves more spatial detail than 2.0 miles.
